In [49]:
import pandas as pd
p = pd.read_csv(ROOT / "data" / "penguins_raw.csv")
p[p["Comments"] == "Not enough blood for isotopes."][["Sex", "Delta 15 N (o/oo)", "Delta 13 C (o/oo)"]]

,Sex,Delta 15 N (o/oo),Delta 13 C (o/oo)
0,MALE,NaN,NaN
12,FEMALE,NaN,NaN
13,MALE,NaN,NaN
15,FEMALE,NaN,NaN
41,MALE,NaN,NaN
46,MALE,NaN,NaN
182,MALE,NaN,NaN


In [50]:
p[p["Comments"] == "No blood sample obtained for sexing."][["Sex", "Delta 15 N (o/oo)", "Delta 13 C (o/oo)"]]

,Sex,Delta 15 N (o/oo),Delta 13 C (o/oo)
9,NaN,9.13362,-25.09368
10,NaN,8.63243,-25.21315


In [51]:
orders = pd.read_csv(ROOT / "data" / "orders_joined.csv")
pd.DataFrame({"type": orders.dtypes.astype(str),
 "distinct": orders.nunique(),
 "missing": orders.isna().sum()})

,type,distinct,missing
order_id,str,1996,0
order_date,str,362,0
customer_id,int64,907,0
channel,str,2,0
category,str,6,0
item_price,float64,1829,0
quantity,int64,5,0
discount_pct,int64,8,0
order_value,float64,1872,0
payment_method,str,4,0


In [52]:
penguins = pd.read_csv(ROOT / "data" / "penguins_raw.csv")
pd.DataFrame({"type": penguins.dtypes.astype(str),
 "distinct": penguins.nunique(),
 "missing": penguins.isna().sum()})

,type,distinct,missing
studyName,str,3,0
Sample Number,int64,152,0
Species,str,3,0
Region,str,1,0
Island,str,3,0
Stage,str,1,0
Individual ID,str,190,0
Clutch Completion,str,2,0
Date Egg,str,50,0
Culmen Length (mm),float64,164,2


In [53]:
print(p.duplicated(["studyName", "Individual ID"]).sum())
print(p.duplicated(["studyName", "Sample Number"]).sum())
print(p.duplicated(["studyName", "Species", "Sample Number"]).sum())

0
124
0


In [54]:
import pandas as pd

full = pd.read_csv(ROOT / "data" / "orders_joined.csv")
exp = full.item_price * full.quantity * (1 - full.discount_pct / 100)

exact = (full.order_value != exp).sum()
penny = ((full.order_value - exp).abs() >= 0.01).sum()

print("flagged by ==            ", exact)
print("flagged within a penny   ", penny)
print("rounding only            ", exact - penny)

flagged by ==             741
flagged within a penny    12
rounding only             729


In [55]:
import pandas as pd

full = pd.read_csv(ROOT / "data" / "orders_joined.csv")
exp = full.item_price * full.quantity * (1 - full.discount_pct / 100)
checks = pd.DataFrame({
    "quantity": full.quantity < 1,
    "price": full.item_price <= 0,
    "delivery": full.delivery_days.notna() & ~full.delivery_days.between(0, 60),
    "value": full.order_value < 0,
    "identity": (full.order_value - exp).abs() >= 0.01,
})
bad = checks[checks.any(axis=1)]
print(bad.sum())
print("delivery is exactly -999:", (full.delivery_days == -999).sum())
print(pd.crosstab(bad.drop(columns="identity").any(axis=1), bad["identity"],
                  rownames=["fails a range rule"], colnames=["fails identity"]))
full.loc[bad.index, ["order_id", "item_price", "quantity", "discount_pct",
                     "order_value", "delivery_days"]].join(bad.add_prefix("fails_"))

quantity     4
price        4
delivery     5
value        4
identity    12
dtype: int64
delivery is exactly -999: 5
fails identity      False  True 
fails a range rule              
True                    5     12


,order_id,item_price,quantity,discount_pct,order_value,delivery_days,fails_quantity,fails_price,fails_delivery,fails_value,fails_identity
178,ORD-100182,353.98,2,10,637.16,-999,False,False,True,False,False
229,ORD-100234,43.71,1,20,-34.97,5,False,False,False,True,True
243,ORD-100250,14.63,1,0,14.63,-999,False,False,True,False,False
309,ORD-100320,0.00,2,0,79.20,5,False,True,False,False,True
364,ORD-100378,69.17,2,0,-138.34,3,False,False,False,True,True
420,ORD-100438,0.00,2,0,18.56,6,False,True,False,False,True
425,ORD-100443,93.46,0,0,186.92,4,True,False,False,False,True
613,ORD-100647,0.00,1,0,62.26,2,False,True,False,False,True
691,ORD-100729,56.62,1,10,50.96,-999,False,False,True,False,False
1069,ORD-101124,133.38,1,0,133.38,-999,False,False,True,False,False


In [56]:
p = pd.read_csv(ROOT / "data" / "penguins_raw.csv")
n_species = p.groupby("Individual ID")["Species"].nunique()
two = n_species[n_species > 1].index
print("IDs with two species:", len(two), " rows carrying them:", p["Individual ID"].isin(two).sum())

IDs with two species: 94  rows carrying them: 228


In [57]:
full.loc[full.order_value < 0, ["order_id", "returned"]]

,order_id,returned
229,ORD-100234,1
364,ORD-100378,0
1519,ORD-101594,0
1782,ORD-101872,0


In [58]:
cols = ["Culmen Length (mm)", "Culmen Depth (mm)", "Flipper Length (mm)",
        "Body Mass (g)", "Sex", "Delta 15 N (o/oo)", "Delta 13 C (o/oo)"]
inc = p[p[cols].isna().any(axis=1)]
print("incomplete rows:", len(inc))
print(inc["studyName"].value_counts())
inc.groupby(inc["Comments"].fillna("(no comment)"))[cols].apply(lambda g: g.isna().sum())

incomplete rows: 20
studyName
PAL0708    15
PAL0910     4
PAL0809     1
Name: count, dtype: int64


,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo)
Comments,,,,,,,
Adult not sampled.,1,1,1,1,1,1,1
Adult not sampled. Nest never observed with full clutch.,1,1,1,1,1,1,1
Nest never observed with full clutch. Not enough blood for isotopes.,0,0,0,0,0,1,1
No blood sample obtained for sexing.,0,0,0,0,2,0,0
No blood sample obtained.,0,0,0,0,2,2,2
No delta15N data received from lab.,0,0,0,0,0,1,0
Not enough blood for isotopes.,0,0,0,0,0,7,7
Sexing primers did not amplify.,0,0,0,0,4,0,0
Sexing primers did not amplify. Not enough blood for isotopes.,0,0,0,0,1,1,1


In [59]:
blank = full[full.marketing_opt_in.isna()]
print("orders:", len(blank), " customers:", blank.customer_id.nunique())
print(blank.channel.value_counts())
print("return rate, blank opt-in:", round(blank.returned.mean(), 3),
      " everyone else:", round(full[full.marketing_opt_in.notna()].returned.mean(), 3))

orders: 77  customers: 48
channel
web    77
Name: count, dtype: int64
return rate, blank opt-in: 0.182  everyone else: 0.192


In [60]:
print(len(full))
print(full.isna().sum()[full.isna().sum() > 0])

1996
customer_region         34
customer_tenure_days    34
marketing_opt_in        77
dtype: int64


In [61]:
print((full.marketing_opt_in.isna() & full.customer_region.notna()).sum())

43


In [62]:
crm_blank = full[full.marketing_opt_in.isna() & full.customer_region.notna()]
print(len(crm_blank), crm_blank.customer_id.nunique(), crm_blank.channel.value_counts().to_dict(),
      round(crm_blank.returned.mean(), 3))

43 21 {'web': 43} 0.116


In [63]:
w1 = pd.read_csv(ROOT / "data" / "retail_orders_week1.csv")
ids = full.loc[full.marketing_opt_in.isna() & full.customer_region.notna(), "customer_id"].unique()
filled = w1[w1.customer_id.isin(ids)]
print("Week 1 rows for these customers:", len(filled))
print(filled.marketing_opt_in.value_counts().to_dict())
per_cust = filled.groupby("customer_id")["marketing_opt_in"].nunique()
print("customers with more than one order:", (filled.customer_id.value_counts() > 1).sum())
print("customers whose orders disagree:", (per_cust > 1).sum())
print("opt-in share where the CRM has a value:",
      full.marketing_opt_in.value_counts(normalize=True).round(3).to_dict())

Week 1 rows for these customers: 43
{True: 33, False: 10}
customers with more than one order: 15
customers whose orders disagree: 0
opt-in share where the CRM has a value: {True: 0.564, False: 0.436}


In [64]:
import pandas as pd

raw = pd.read_csv(ROOT / "data" / "orders_joined.csv")
clean = pd.read_csv(ROOT / "outputs" / "orders_clean.csv")

ids = raw.loc[raw["order_value"] < 0, "order_id"]
before = raw[raw["order_id"].isin(ids)].set_index("order_id")
after = clean[clean["order_id"].isin(ids)].set_index("order_id")

check = pd.DataFrame({
    "returned": before["returned"],
    "value_before": before["order_value"],
    "value_after": after["order_value"],
    "sign_corrected": after["order_value_sign_corrected"],
    "ambiguous": after["order_value_ambiguous"],
    "identity": (before["item_price"] * before["quantity"]
                 * (1 - before["discount_pct"] / 100)).round(2),
})
check["magnitude_matches"] = (check["value_before"].abs() - check["identity"]).abs() <= 0.01
check

,returned,value_before,value_after,sign_corrected,ambiguous,identity,magnitude_matches
order_id,,,,,,,
ORD-100234,1,-34.97,34.97,True,True,34.97,True
ORD-100378,0,-138.34,138.34,True,False,138.34,True
ORD-101594,0,-64.04,64.04,True,False,64.04,True
ORD-101872,0,-21.27,21.27,True,False,21.27,True


In [65]:
o = pd.read_csv(ROOT / "data" / "orders_joined.csv")
filled = o["marketing_opt_in"].fillna(o["marketing_opt_in"].mode()[0])
print("as recorded:", o["marketing_opt_in"].value_counts().to_dict(), "unknown", o["marketing_opt_in"].isna().sum())
print("filled:     ", filled.value_counts().to_dict())
print(f"opted-in share {o['marketing_opt_in'].mean():.3f} -> {filled.mean():.3f}")
print(o.loc[o["marketing_opt_in"].isna(), "customer_id"].nunique())

as recorded: {True: 1083, False: 836} unknown 77
filled:      {True: 1160, False: 836}
opted-in share 0.564 -> 0.581
48


In [66]:
import json
from pathlib import Path
import pandas as pd

# works whether the notebook sits in notebooks/ or in the repository root
ROOT = Path.cwd() if (Path.cwd() / "outputs").exists() else Path.cwd().parent
feat = json.loads((ROOT / "outputs" / "features.json").read_text())
enc = json.loads((ROOT / "outputs" / "encoding_lab.json").read_text())
orders = pd.read_csv(ROOT / "data" / "orders_clean.csv")

def s(x):
    return f"{x['mean']:.4f} (sd {x['sd']:.4f})"

print(f"orders modelled: {len(orders)}, customers: {orders['customer_id'].nunique()}")

print("\n## The six date features")
w, d = feat["auc_without_date_features"], feat["auc_with_date_features"]
diff = d["mean"] - w["mean"]
print(f"without: {s(w)}   with: {s(d)}")
print(f"difference {diff:+.4f}, which is {'smaller' if abs(diff) < w['sd'] else 'larger'} "
      f"than the sd of {w['sd']:.4f}")

print("\n## The three routes (paste into the table)")
names = {"one_hot_only": "One-hot only, no customer rate",
         "target_encoded_in_the_pipeline": "Customer target encoded inside the pipeline",
         "customer_rate_on_every_order": "Customer rate on every order, before the split"}
for key, label in names.items():
    r = enc["scores"][key]
    print(f"| {label} | {s(r['random_split'])} | {s(r['grouped_by_customer'])} |")

print("\n## Correlation with own label (paste into the table)")
labels = {"group_mean_on_every_order": "Group mean on every order",
          "target_encoder_fit_then_transform": "TargetEncoder, fit then transform",
          "target_encoder_fit_transform": "TargetEncoder, fit_transform (cross-fitted)"}
for key, label in labels.items():
    print(f"| {label} | {enc['correlation_with_own_label'][key]:.4f} |")

print("\n## Rows per level (paste into the table)")
for col, v in sorted(enc["rows_per_level"].items(), key=lambda kv: kv[1]["values"]):
    print(f"| {col} | {v['values']} | {v['orders_per_value']:.1f} |")

print("\n## The ambiguous negative order_value")
if "order_value_ambiguous" in orders.columns:
    amb = orders[orders["order_value_ambiguous"].astype(bool)]
    print(amb[["order_id", "order_value", "returned"]].to_string(index=False))
else:
    print("no order_value_ambiguous column in data/orders_clean.csv")

orders modelled: 1996, customers: 907

## The six date features
without: 0.6199 (sd 0.0347)   with: 0.6120 (sd 0.0376)
difference -0.0079, which is smaller than the sd of 0.0347

## The three routes (paste into the table)
| One-hot only, no customer rate | 0.6199 (sd 0.0347) | 0.6142 (sd 0.0296) |
| Customer target encoded inside the pipeline | 0.6223 (sd 0.0340) | 0.6145 (sd 0.0294) |
| Customer rate on every order, before the split | 0.9326 (sd 0.0102) | 0.9335 (sd 0.0149) |

## Correlation with own label (paste into the table)
| Group mean on every order | 0.6880 |
| TargetEncoder, fit then transform | 0.6802 |
| TargetEncoder, fit_transform (cross-fitted) | 0.0630 |

## Rows per level (paste into the table)
| channel | 2 | 998.0 |
| payment_method | 4 | 499.0 |
| category | 6 | 332.7 |
| customer_region | 7 | 285.1 |
| customer_id | 907 | 2.2 |

## The ambiguous negative order_value
  order_id  order_value  returned
ORD-100234        34.97         1


In [67]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "outputs").exists() else Path.cwd().parent
orders = pd.read_csv(ROOT / "data" / "orders_clean.csv")

returned = orders[orders["returned"] == 1]
print(f"{len(returned)} returned orders out of {len(orders)} "
      f"({returned.shape[0] / len(orders):.1%})")

pd.set_option("display.max_rows", None)      # show every row, not a truncated view
returned

382 returned orders out of 1996 (19.1%)


,order_id,order_date,customer_id,channel,category,item_price,quantity,discount_pct,order_value,payment_method,...,customer_prior_orders,returned,customer_region,customer_tenure_days,marketing_opt_in,quantity_missing,item_price_missing,order_value_sign_corrected,order_value_ambiguous,delivery_days_missing
4,ORD-100004,2025-12-09,10369,web,apparel,39.69,3.0,15,101.21,card,...,2,1,London,453.0,True,False,False,False,False,False
12,ORD-100013,2025-10-19,10074,web,electronics,402.95,1.0,10,362.66,bnpl,...,0,1,South,1026.0,True,False,False,False,False,False
16,ORD-100017,2025-11-25,10128,web,electronics,113.11,1.0,0,113.11,paypal,...,1,1,Midlands,327.0,True,False,False,False,False,False
17,ORD-100018,2025-03-09,10144,web,home,62.10,1.0,0,62.10,paypal,...,0,1,Midlands,1114.0,True,False,False,False,False,False
23,ORD-100024,2025-10-29,10653,web,footwear,48.04,1.0,0,48.04,paypal,...,1,1,South,1092.0,False,False,False,False,False,False
24,ORD-100025,2025-10-21,10611,web,apparel,25.73,1.0,0,25.73,card,...,1,1,London,1876.0,True,False,False,False,False,False
25,ORD-100026,2025-08-31,10375,web,apparel,71.87,1.0,15,61.09,card,...,3,1,Midlands,868.0,True,False,False,False,False,False
34,ORD-100035,2025-05-01,10556,web,footwear,40.16,1.0,0,40.16,card,...,0,1,London,2254.0,True,False,False,False,False,False
46,ORD-100047,2025-04-19,10874,web,electronics,167.52,1.0,5,159.14,card,...,0,1,North,2200.0,True,False,False,False,False,False
47,ORD-100048,2025-10-07,10815,web,apparel,60.79,1.0,0,60.79,paypal,...,2,1,South,1104.0,True,False,False,False,False,False


In [68]:
pd.read_csv(ROOT / "data" / "orders_clean.csv").query("order_value < 0")

,order_id,order_date,customer_id,channel,category,item_price,quantity,discount_pct,order_value,payment_method,...,customer_prior_orders,returned,customer_region,customer_tenure_days,marketing_opt_in,quantity_missing,item_price_missing,order_value_sign_corrected,order_value_ambiguous,delivery_days_missing
